# CodeT5 GPU fallback: a fresh reproduction without lab SSH

Choose Runtime → Change runtime type → T4 GPU in Colab, if a free T4 is available. No paid purchase is selected here. This notebook rebuilds the LSTM crafter and the clean, 5%, 10% and 20% CodeT5 checkpoints, then measures the two feature detectors.

**Newly trained checkpoints must be labeled “fresh Colab reproduction.” They are not the saved lab models or their 97.84% result.** Training may exceed free runtime capacity. This notebook has no saved outputs or invented rates.

Before training, an audited staged copy omits the unrelated reranking ablation, fixes one-beam decoding for the matched comparison, and schedules every budget regardless of relative validation attack success. Original repository scripts are unchanged. Current validation quality flags remain measured, not assumed.

## 1. Configure and inspect the runtime

The kernel orchestrates jobs; an isolated Python 3.10 runs the legacy pinned project stack. Leave PROJECT_REVISION empty to record the exact branch checkout, or enter the full published fallback commit. An existing source checkout is reused without pulling or overwriting changes. Optional prepared-data upload avoids the full public dataset download.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, shutil, subprocess, sys, zipfile
PROJECT_URL = "https://github.com/Jas-thinks/lstm-code-backdoor-detection.git"
PROJECT_BRANCH = "codex/codet5-detection-benchmark"
PROJECT_REVISION = ""  # Optional full published fallback commit.
MINIMUM_SOURCE_COMMIT = "33d98c386fe37d62ca4dd813a3b4247b54a02110"
UV_VERSION = "0.13.8"
USE_PREPARED_UPLOAD = False
EXPECTED_PREPARED_SHA256 = "65e1eff6d234b0f1bf27868d5d7d0d82097b130915c4561514d470892b8a46fa"
WORKSPACE = Path("/content/btech-codet5-fallback")
SOURCE = WORKSPACE / "source"
BUNDLE = WORKSPACE / "experiment/lab-run"
RUN_ROOT = BUNDLE / "run"  # Follow-up scripts require this layout.
PYTHON = RUN_ROOT / "environment/bin/python"
if platform.system() != "Linux" or not Path("/content").is_dir() or shutil.which("nvidia-smi") is None:
    raise RuntimeError("Use a Colab Linux T4 GPU runtime; local CPU work has a separate runner.")
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                               "--format=csv,noheader"], text=True).strip()
if len(gpu.splitlines()) != 1 or "T4" not in gpu:
    raise RuntimeError(f"This CUDA 11.7 recipe is prepared for T4, not this allocation: {gpu}")
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("GPU:", gpu)
print("Fresh reproduction; no saved lab checkpoint or rate has been loaded.")

## 2. Fetch source and stage the declared protocol

uv is installed as a wheel from official PyPI and obtains Python 3.10 through its managed Python support. No downloaded shell installer is executed. The helper verifies known original source hashes and exact patch markers before writing a separate staged copy. Source, adaptations and protocol are recorded before training.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--index-url", "https://pypi.org/simple", "--only-binary=:all:", f"uv=={UV_VERSION}"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv executable is not on PATH after installation.")
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", PROJECT_BRANCH, PROJECT_URL, str(SOURCE)], check=True)
if subprocess.check_output(["git", "-C", str(SOURCE), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Source has local changes; inspect them or choose a fresh workspace.")
if PROJECT_REVISION:
    subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", PROJECT_REVISION], check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
subprocess.run(["git", "-C", str(SOURCE), "merge-base", "--is-ancestor", MINIMUM_SOURCE_COMMIT, SOURCE_COMMIT], check=True)
helper = SOURCE / "lab-run/scripts/prepare_portable_reproduction.py"
if not helper.is_file():
    raise RuntimeError("Checkout predates the fallback helper; use the published fallback branch/commit.")
subprocess.run([sys.executable, str(helper), "--source", str(SOURCE / "lab-run"),
                "--destination", str(BUNDLE), "--source-commit", SOURCE_COMMIT], check=True)
subprocess.run([UV, "python", "install", "3.10"], check=True)
if not PYTHON.is_file():
    subprocess.run([UV, "venv", "--seed", "--python", "3.10", str(RUN_ROOT / "environment")], check=True)
ENV = dict(os.environ, AFRAIDOOR_RUN_ROOT=str(RUN_ROOT), PYTHONDONTWRITEBYTECODE="1", CUDA_VISIBLE_DEVICES="0")
(RUN_ROOT / "notebook-logs").mkdir(parents=True, exist_ok=True)
context = {"kind":"fresh_colab_reproduction", "source_commit":SOURCE_COMMIT, "gpu":gpu,
           "uses_saved_lab_checkpoints":False, "orchestrator_python":sys.version,
           "uv":subprocess.check_output([UV,"--version"],text=True).strip(), "prepared_archive_sha256":None}
(RUN_ROOT / "notebook-context.json").write_text(json.dumps(context, indent=2) + "\n")
print("Source commit:", SOURCE_COMMIT)
print((BUNDLE / "portable-protocol.json").read_text())

## 3. Optional prepared-data upload

When USE_PREPARED_UPLOAD is true, upload exactly prepared-seed42.tar.gz and prepared-seed42.sha256 from the supplied fallback package. Both are checked against the known archive digest. The runner checks the digest again before extraction. Otherwise the runner downloads pinned public CodeSearchNet data.

In [ ]:
if USE_PREPARED_UPLOAD:
    from google.colab import files
    upload_dir = WORKSPACE / "uploads"
    upload_dir.mkdir(exist_ok=True)
    previous = Path.cwd()
    os.chdir(upload_dir)
    try:
        uploaded = files.upload()
    finally:
        os.chdir(previous)
    if set(uploaded) != {"prepared-seed42.tar.gz", "prepared-seed42.sha256"}:
        raise ValueError("Upload exactly the prepared archive and its checksum.")
    actual = hashlib.sha256(uploaded["prepared-seed42.tar.gz"]).hexdigest()
    checksum = uploaded["prepared-seed42.sha256"].decode("utf-8").split()
    if actual != EXPECTED_PREPARED_SHA256 or checksum != [EXPECTED_PREPARED_SHA256, "prepared-seed42.tar.gz"]:
        raise ValueError("Prepared data/checksum differs from the vetted project asset.")
    (BUNDLE / "prepared-seed42.tar.gz").write_bytes(uploaded["prepared-seed42.tar.gz"])
    (BUNDLE / "prepared-seed42.sha256").write_text(f"{EXPECTED_PREPARED_SHA256}  prepared-seed42.tar.gz\n")
    context["prepared_archive_sha256"] = actual
    (RUN_ROOT / "notebook-context.json").write_text(json.dumps(context, indent=2) + "\n")
    print("Verified prepared archive:", actual)
else:
    print("Using the original pinned public dataset download and preparation path.")

## 4. Install the project environment with logged execution

Logs and stage status stay in the new run directory. GPU jobs use a nonblocking lock and refuse to start if another compute process is using the GPU. CodeT5 supports epoch-level resume; the old LSTM trainer does not automatically resume an interrupted training run. Export artifacts before the free runtime ends.

In [ ]:
def run_recorded(label, arguments):
    log_path = RUN_ROOT / "notebook-logs" / (label + ".log")
    state_path = RUN_ROOT / "notebook-stages.json"
    state = json.loads(state_path.read_text()) if state_path.is_file() else {}
    state[label] = {"status":"running", "command":[str(x) for x in arguments],
                    "started_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()}
    state_path.write_text(json.dumps(state, indent=2) + "\n")
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen([str(x) for x in arguments], cwd=BUNDLE, env=ENV,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        status = process.wait()
    state[label].update(status="completed" if status == 0 else "failed", returncode=status,
                        finished_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
    state_path.write_text(json.dumps(state, indent=2) + "\n")
    if status:
        raise RuntimeError(f"{label} failed ({status}); inspect {log_path} before proceeding.")

LOCKED_RUNNER = """import subprocess, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'scripts'))
from benchmark_detection import gpu_lock
with gpu_lock():
    subprocess.run([sys.executable, '-u', sys.argv[1]], check=True)
"""
def run_locked(label, script):
    run_recorded(label, [PYTHON, "-u", "-c", LOCKED_RUNNER, str(BUNDLE / script)])

run_recorded("data_and_environment", ["bash", "run.sh", "data"])

## 5. Check hardware compatibility and implementation

These checks do not supply project performance results. The probe uses the isolated Python, not the notebook kernel's preinstalled PyTorch.

In [ ]:
probe = """import torch, transformers
assert torch.__version__ == '1.13.1+cu117', torch.__version__
assert transformers.__version__ == '4.26.1', transformers.__version__
assert torch.cuda.is_available()
assert torch.cuda.get_device_capability(0) == (7, 5), 'Expected T4 capability 7.5'
assert 'sm_75' in torch.cuda.get_arch_list(), torch.cuda.get_arch_list()
x = torch.ones((32, 32), device='cuda')
assert torch.isfinite(x @ x).all().item()
print('GPU probe passed:', torch.cuda.get_device_name(0), torch.__version__, transformers.__version__)
"""
run_recorded("cuda_probe", [PYTHON, "-c", probe])
run_recorded("regression_tests", [PYTHON, "-m", "unittest", "discover", "-s", str(SOURCE / "lab-run/tests"), "-v"])

## 6. Train the crafter and clean/5% models

The staged initial pipeline retains data and trigger audits, clean/poisoned training and the original baseline diagnostic evaluation. It omits the unrelated top-eight reranking models/comparison. The final matched report uses one beam for every planned model.

In [ ]:
run_recorded("clean_and_5pct_models", ["bash", "run.sh"])

## 7. Measure the validation sweep with a predeclared one-beam decoder

The four validation settings remain recorded. The report preserves the historical rule's candidate, the measured quality flag, and the separate fixed one-beam protocol. It does not claim that one beam was best in this new run.

In [ ]:
run_locked("one_beam_validation", "scripts/decode_sweep.py")

## 8. Train/evaluate the 10% model

The original training, input identity and nesting checks remain active. All rate and quality fields come from this run.

In [ ]:
run_locked("10pct_model", "followups/poison-budget-10pct/run_experiment.py")

## 9. Train/evaluate the 20% model

The old observed 299 validation hits are not asserted. This planned budget is evaluated even when its validation attack rate does not improve. Measured quality failures remain visible.

In [ ]:
run_locked("20pct_model", "followups/poison-budget-20pct/run_experiment.py")

## 10. Matched benchmark and detector evaluation

Only a fully successful execution produces a completed report. Input/checkpoint identity checks, training-only detector fitting and clean-only calibration remain active. Keep this evidence separate from the saved lab experiment.

In [ ]:
if "BENCHMARK_OUT" not in globals():
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%fZ")
    BENCHMARK_OUT = RUN_ROOT / "benchmark-detection" / ("fresh-colab-" + stamp)
run_recorded("benchmark_and_detection", [PYTHON, "-u", "scripts/benchmark_detection.py",
                                         "--run-root", RUN_ROOT, "--out", BENCHMARK_OUT])
report_path = BENCHMARK_OUT / "codet5-benchmark-detection.json"
report = json.loads(report_path.read_text())
if report.get("completed") is not True or len(report.get("benchmark_models", {})) != 4 or len(report.get("detectors", {})) != 2:
    raise RuntimeError("Every planned measurement has not completed.")
report["reproduction"] = {
    "kind":"fresh_colab_reproduction", "uses_saved_lab_checkpoints":False, "source_commit":SOURCE_COMMIT,
    "portable_protocol_sha256":hashlib.sha256((BUNDLE / "portable-protocol.json").read_bytes()).hexdigest(),
    "context":context, "interpretation":"New weights/triggers; distinct from the saved 97.84% lab experiment."}
report_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
shutil.copy2(BUNDLE / "portable-protocol.json", BENCHMARK_OUT / "portable-protocol.json")
print("Completed new report:", report_path)

## 11. Inspect measured values

Higher pre-defense attack ASR demonstrates the attack. Detector recall/F1 and clean false positives measure detection; post-filter ASR and clean retention measure defense. Interpret them separately.

In [ ]:
from IPython.display import display, Markdown
report = json.loads((BENCHMARK_OUT / "codet5-benchmark-detection.json").read_text())
if report.get("completed") is not True:
    raise RuntimeError("No complete report is available.")
lines = ["### Fresh Colab reproduction", "| Model | Triggered hits | ASR | Clean BLEU-4 | Clean target hits |",
         "|---|---:|---:|---:|---:|"]
for name, value in report["benchmark_models"].items():
    lines.append(f"| {name} | {value['triggered_target_hits']}/{value['triggered_rows']} | {value['asr_percent']:.2f}% | {value['clean_bleu4']:.4f} | {value['full_clean_target_hits']}/{value['clean_rows']} |")
display(Markdown("\n".join(lines)))
lines = ["### Detection on the new 20% model", "| Detector | Recall | F1 | Clean FPR | Post-filter ASR | Clean retention |",
         "|---|---:|---:|---:|---:|---:|"]
for name, value in report["detectors"].items():
    metric, after = value["metrics"], value["after_filter"]
    lines.append(f"| {name} | {100*metric['recall']:.2f}% | {100*metric['f1']:.2f}% | {100*metric['clean_false_positive_rate']:.2f}% | {after['asr_percent']:.2f}% | {after['clean_retention_percent']:.2f}% |")
display(Markdown("\n".join(lines)))
print("Inspect paired_metrics, individual predictions/scores and actual validation quality flags too.")

## 12. Export evidence and optional selected checkpoints

Run before the free runtime ends; it can also export partial logs. Set EXPORT_TRAINED_CHECKPOINTS to create a potentially large archive of selected LSTM/CodeT5 models and their metadata. This is a model/inference backup, not a complete optimizer-resume backup.

Only the evidence ZIP downloads by default. Drive backup is disabled: enable SAVE_TO_DRIVE only if you choose to authorize your own Drive. No Drive mounting or sharing change occurs otherwise.

In [ ]:
import tarfile
EXPORT_TRAINED_CHECKPOINTS = False
SAVE_TO_DRIVE = False
DOWNLOAD_EVIDENCE = True
exports = WORKSPACE / "exports"
exports.mkdir(exist_ok=True)
evidence_path = exports / "fresh-colab-codet5-evidence.zip"
with zipfile.ZipFile(evidence_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    candidates = [BUNDLE / "portable-protocol.json", RUN_ROOT / "notebook-context.json", RUN_ROOT / "notebook-stages.json"]
    for folder in (RUN_ROOT / "notebook-logs", RUN_ROOT / "benchmark-detection", RUN_ROOT / "decode-sweep"):
        if folder.is_dir():
            candidates.extend(path for path in folder.rglob("*") if path.is_file())
    for folder in (RUN_ROOT / "baseline", RUN_ROOT / "poison-budget-10pct", RUN_ROOT / "poison-budget-20pct"):
        if folder.is_dir():
            candidates.extend(folder.glob("*.json"))
    for path in sorted(set(candidates)):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(BUNDLE)))
print("Evidence ZIP:", evidence_path, evidence_path.stat().st_size, "bytes")
backup_path = None
if EXPORT_TRAINED_CHECKPOINTS:
    checkpoints = set()
    for folder, name in ((RUN_ROOT / "baseline","latest_clean_victim.json"),
                         (RUN_ROOT / "baseline","latest_poisoned_victim.json"),
                         (RUN_ROOT / "poison-budget-10pct","latest_poisoned_victim.json"),
                         (RUN_ROOT / "poison-budget-20pct","latest_poisoned_victim.json")):
        manifest = folder / name
        if manifest.is_file():
            checkpoints.add(Path(json.loads(manifest.read_text())["best_checkpoint"]))
    lstm = RUN_ROOT / "shared/latest_lstm_run.json"
    if lstm.is_file():
        checkpoints.add(Path(json.loads(lstm.read_text())["selected_checkpoint"]))
    if not checkpoints:
        raise RuntimeError("No selected trained checkpoint is available yet.")
    backup_path = exports / "fresh-colab-selected-checkpoints.tar.gz"
    with tarfile.open(backup_path, "w:gz") as archive:
        for path in sorted(checkpoints):
            archive.add(path, arcname=str(path.resolve().relative_to(BUNDLE.resolve())))
        for folder in (RUN_ROOT / "baseline", RUN_ROOT / "poison-budget-10pct", RUN_ROOT / "poison-budget-20pct", RUN_ROOT / "shared"):
            if folder.is_dir():
                for path in sorted(folder.glob("latest_*.json")):
                    archive.add(path, arcname=str(path.relative_to(BUNDLE)))
    print("Selected-model backup:", backup_path, backup_path.stat().st_size, "bytes")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive/BTech-CodeT5-fresh-reproduction")
    target.mkdir(parents=True, exist_ok=True)
    for path in (evidence_path, backup_path):
        if path is not None:
            shutil.copy2(path, target / path.name)
    print("Chosen exports saved to your Drive:", target)
if DOWNLOAD_EVIDENCE:
    from google.colab import files
    files.download(str(evidence_path))
print("Download the optional large checkpoint archive from the Files panel if created; nothing is pushed to GitHub.")

## Interpretation and setup references

Keep the source commit, portable protocol, prediction/score files and report together. No 90% attack/detection rate is promised. This is one seed, a reused Python test set, trusted-clean reference data and one trigger family. New LSTM weights can change crafted identifiers.

The separate local pretrained benchmark does not replace these fine-tuned/poisoned model measurements. This notebook has been authored and checked statically; completing its GPU run is necessary before reporting new effectiveness.

[uv installation](https://docs.astral.sh/uv/getting-started/installation/), [managed Python](https://docs.astral.sh/uv/guides/install-python/), [PyTorch previous versions](https://pytorch.org/get-started/previous-versions/), [TA protocol](https://github.com/Jas-thinks/lstm-code-backdoor-detection/blob/codex/codet5-detection-benchmark/docs/TA-REQUIREMENTS.md).